# Phenology and bloom prediction

This notebook combines hourly temperature-derived chill/heat metrics with simple bloom
requirement scenarios, then compares predicted bloom dates with synthetic observations.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from chillPy import bloom_prediction2, chilling_hourtable, stack_hourly_temps

DATA = Path("examples/data")
weather = pd.read_csv(DATA / "synthetic_daily_weather.csv")
pheno = pd.read_csv(DATA / "synthetic_phenology.csv")
pheno

In [ ]:
hourly = stack_hourly_temps(weather.query("2015 <= Year <= 2022"), latitude=42.0)["hourtemps"]
chill_table = chilling_hourtable(hourly, start_jday=305)

predictions = bloom_prediction2(
    chill_table,
    chill_req=[38],
    heat_req=[5200],
    chill_model="Chill_Portions",
    heat_model="GDH",
    start_jday=305,
)
predictions.head()

In [ ]:
comparison = pheno.merge(
    predictions[["Season", "Pheno_date"]].rename(columns={"Season": "Year", "Pheno_date": "predicted"}),
    on="Year",
    how="inner",
)
comparison["error_days"] = comparison["predicted"] - comparison["pheno"]
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(comparison["Year"], comparison["pheno"], marker="o", label="observed")
ax.plot(comparison["Year"], comparison["predicted"], marker="s", label="predicted")
ax.set_ylabel("Bloom day of year")
ax.set_title("Synthetic bloom-date prediction")
ax.legend(frameon=False)
fig